In [20]:
import mne 
import torch
import numpy as np
import os 
from sklearn.model_selection import train_test_split

In [21]:
np.random.seed(42)  # 再現性（ダウンサンプリングの乱数固定)

In [22]:
subject_ids = range(1,110)
all_x,all_y = [],[]

In [23]:
base_path = "E:/EEG_Dataset/eeg-motor-movementimagery-dataset-1.0.0"

In [24]:
for subject_id in subject_ids:
    subj = f"S{subject_id:03d}"
    
    for run in ["R04", "R08", "R12"]:
        edf_path = os.path.join(base_path, subj, f"{subj}{run}.edf")
        
        

        raw = mne.io.read_raw_edf(edf_path, preload=True)
        raw.rename_channels(lambda s: s.replace('.','').upper())
        raw.pick_channels([
           "FC1", "FC2", "C3", "CZ", "C4", "CP1", "CP2", "PZ"
        ])
        
        raw.filter(1., 30., fir_design='firwin')

        events, _ = mne.events_from_annotations(raw)

        event_id = {'T1': 1, 'T2': 2}

        epochs = mne.Epochs(raw, events, event_id=event_id,
                                tmin=-0.5, tmax=3.5,
                                baseline=None, preload=True)

        x = epochs.get_data()
        y_ids = epochs.events[:, 2]    # 1 or 2 が入ってる
        y=np.where(y_ids == event_id['T1'], 0, 1) #T1→0, T2→1
        
        all_x.append(x)
        all_y.append(y)

Extracting EDF parameters from E:\EEG_Dataset\eeg-motor-movementimagery-dataset-1.0.0\S001\S001R04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 529 samples (3.306 s)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Not setting metadata
23 matching event

C:\Users\kgtk0\AppData\Local\Temp\ipykernel_28400\3225885258.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True)
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_28400\3225885258.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True)
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_28400\3225885258.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 529 samples (3.306 s)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 23 events and 641 original time points ...
1 bad epochs dropped
Extracting EDF parameters from E:\EEG_Dataset\eeg-motor-movementimagery-dataset

In [25]:
all_x_fixed = []
all_y_fixed = []
for x, y in zip(all_x, all_y):
    if x.shape[2] == 641:
        all_x_fixed.append(x)
        all_y_fixed.append(y)

x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)


print("all:", x_all.shape, y_all.shape)  # 例：(試行数, 8チャネル, 時間点数)

all: (6855, 8, 641) (6855,)


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

In [27]:
#学習データのクラスバランス調整
idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]
min_len = min(len(idx0), len(idx1))
idx0_ds = np.random.choice(idx0, min_len, replace=False)
idx1_ds = np.random.choice(idx1, min_len, replace=False)
balanced_idx = np.concatenate([idx0_ds, idx1_ds])

X_train = X_train[balanced_idx]
y_train = y_train[balanced_idx]

In [28]:
# 正規化
X_train = (X_train - X_train.mean(axis=2, keepdims=True)) / X_train.std(axis=2, keepdims=True)
X_test  = (X_test  - X_test.mean(axis=2, keepdims=True))  / X_test.std(axis=2, keepdims=True)

In [29]:
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, C, T)
        self.y = torch.tensor(y, dtype=torch.long)     # (N,)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [30]:
from torch.utils.data import DataLoader

train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [31]:
import torch.nn as nn
import torch.nn.functional as F

class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=16, samples=641):
        super(EEGNet, self).__init__()

        # 第1層: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        # 第2層: Depthwise Convolution (Spatial filter)
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(p=0.25)
        )

        # 第3層: Separable Convolution
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(p=0.25)
        )

        # 全結合層
        self.classify = nn.Linear(self._get_flattened_size(channels, samples), num_classes)

    def _get_flattened_size(self, channels, samples):
        x = torch.zeros(1, 1, channels, samples)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        return x.numel()

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, C, T)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        x = x.view(x.size(0), -1)
        return self.classify(x)

In [32]:
device = torch.device("cuda")  

model = EEGNet(num_classes=2, channels=8, samples=641).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


In [33]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    acc = correct / total
    return running_loss / len(loader), acc

def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    return correct / total

In [34]:
epochs = 50  # 時間と相談して増減してOK

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

Epoch 1/50 - Train Loss: 0.6076, Train Acc: 0.6692, Test Acc: 0.6885
Epoch 2/50 - Train Loss: 0.5520, Train Acc: 0.7264, Test Acc: 0.7520
Epoch 3/50 - Train Loss: 0.5278, Train Acc: 0.7425, Test Acc: 0.7535
Epoch 4/50 - Train Loss: 0.4903, Train Acc: 0.7653, Test Acc: 0.7746
Epoch 5/50 - Train Loss: 0.4921, Train Acc: 0.7679, Test Acc: 0.7622
Epoch 6/50 - Train Loss: 0.4640, Train Acc: 0.7796, Test Acc: 0.7578
Epoch 7/50 - Train Loss: 0.4623, Train Acc: 0.7851, Test Acc: 0.7848
Epoch 8/50 - Train Loss: 0.4674, Train Acc: 0.7773, Test Acc: 0.7841
Epoch 9/50 - Train Loss: 0.4538, Train Acc: 0.7869, Test Acc: 0.7797
Epoch 10/50 - Train Loss: 0.4474, Train Acc: 0.7879, Test Acc: 0.7899
Epoch 11/50 - Train Loss: 0.4442, Train Acc: 0.7892, Test Acc: 0.7797
Epoch 12/50 - Train Loss: 0.4289, Train Acc: 0.7978, Test Acc: 0.7819
Epoch 13/50 - Train Loss: 0.4224, Train Acc: 0.8043, Test Acc: 0.7965
Epoch 14/50 - Train Loss: 0.4248, Train Acc: 0.7981, Test Acc: 0.7848
Epoch 15/50 - Train Loss: 0.4

In [35]:
# モデルの重み保存
torch.save(model.state_dict(), 'eegnet_model.pt')

In [36]:
np.bincount(y_train)

array([1924, 1924])